# LLM Judge Query Refinement Evaluation

This notebook evaluates the additional LLM judge step added after the first BM25 + cross-encoder pass.

The pipeline flow is:
1. Load or generate the four query variants.
2. Run BM25 and weighted RRF over the original query plus query variants.
3. Rerank the weighted RRF candidates with the cross-encoder using the original query.
4. Give the LLM judge the top 15 original BM25 results and top 15 weighted RRF + CE results.
5. Let the judge refine the four query variants for higher recall.
6. Run weighted RRF and cross-encoder again with the judged query variants.

The most important output sheet is `BM25_RRF_weighted_CE_judged`.

## Setup

This cell finds the project root, switches the working directory, and makes sure local modules such as `main.py` can be imported.

In [11]:
from pathlib import Path
import importlib
import os
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
while project_root.name != "Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval" and project_root.parent != project_root:
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Working directory: {Path.cwd()}")

Working directory: /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval


## Run The Judged Pipeline

Set `RUN_PIPELINE = True` when you want to execute the full pipeline. The default is `False` so opening the notebook does not immediately spend LLM tokens.

Use `PROVIDER = "records"` to reuse the existing recorded query variants. The judge still needs an LLM, so set `JUDGE_PROVIDER` to `"gemini"` or `"ollama"`.

In [12]:
USE_CASE = "uc2"
PROVIDER = "records"
JUDGE_PROVIDER = "gemini"
RECORDS_PATH = None
RUN_PIPELINE = True

if RUN_PIPELINE:
    import main

    importlib.reload(main)
    main.retrieval_pipeline(
        provider=PROVIDER,
        use_case=USE_CASE,
        records_path=RECORDS_PATH,
        enable_llm_judge=True,
        judge_provider=JUDGE_PROVIDER,
    )
else:
    print("RUN_PIPELINE is False. Set it to True to generate judged retrieval outputs.")

Loaded recorded query variants from output_with_agents_uc2.csv
Loaded 1 queries for level 'process'
[process] Processing row 1/1...
Loaded 7 queries for level 'subprocess'
[subprocess] Processing row 1/7...
[subprocess] Processing row 2/7...
[subprocess] Processing row 3/7...
[subprocess] Processing row 4/7...
[subprocess] Processing row 5/7...
[ERROR] Gemini generation failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The request timed out. Please try again.', 'status': 'UNAVAILABLE'}}
[WARN] LLM judge returned empty response; keeping original query variants.
[subprocess] Processing row 6/7...
[subprocess] Processing row 7/7...
Loaded 19 queries for level 'task'
[task] Processing row 1/19...
[task] Processing row 2/19...
[task] Processing row 3/19...
[task] Processing row 4/19...
[ERROR] Gemini generation failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again lat

## Load Judged Outputs

This cell loads the Excel output generated by the pipeline and shows the judged sheets. If a sheet is empty, rerun the previous cell with `RUN_PIPELINE = True`.

In [13]:
ranking_path = project_root / f"output_ranking_{USE_CASE}.xlsx"
agents_path = project_root / f"output_with_agents_{USE_CASE}.csv"

JUDGED_BM25_SHEET = "BM25_RRF_weighted_judged"
JUDGED_CE_SHEET = "BM25_RRF_weighted_CE_judged"

if not ranking_path.exists():
    raise FileNotFoundError(f"Missing {ranking_path}. Run the pipeline first.")

judged_bm25 = pd.read_excel(ranking_path, sheet_name=JUDGED_BM25_SHEET)
judged_ce = pd.read_excel(ranking_path, sheet_name=JUDGED_CE_SHEET)

print(f"Loaded {ranking_path}")
print(f"Judged BM25 rows: {len(judged_bm25)}")
print(f"Judged CE rows: {len(judged_ce)}")

display(judged_bm25.head(10))
display(judged_ce.head(10))

Loaded /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval/output_ranking_uc2.xlsx
Judged BM25 rows: 595
Judged CE rows: 595


,level,query,rel_text,score,method,query_variant,source_variants
0,process,Know Your Customer (KYC) regulations for indiv...,If a reporting entity is unable to establish t...,0.016393,bm25_rrf_weighted_judged,llm_judge_weighted_rrf,"baseline, judge_legal_terminology_rewrite, jud..."
1,process,Know Your Customer (KYC) regulations for indiv...,Customer due diligence procedures (however des...,0.016129,bm25_rrf_weighted_judged,llm_judge_weighted_rrf,"baseline, judge_legal_terminology_rewrite, jud..."
2,process,Know Your Customer (KYC) regulations for indiv...,An AML/CTF program must include appropriate ri...,0.015059,bm25_rrf_weighted_judged,llm_judge_weighted_rrf,"baseline, judge_legal_terminology_rewrite, jud..."
3,process,Know Your Customer (KYC) regulations for indiv...,An AML/CTF program must include appropriate ri...,0.014425,bm25_rrf_weighted_judged,llm_judge_weighted_rrf,"baseline, judge_legal_terminology_rewrite, jud..."
4,process,Know Your Customer (KYC) regulations for indiv...,An AML/CTF program must include a procedure fo...,0.014142,bm25_rrf_weighted_judged,llm_judge_weighted_rrf,"baseline, judge_legal_terminology_rewrite, jud..."
5,process,Know Your Customer (KYC) regulations for indiv...,An AML/CTF program must include appropriate ri...,0.013956,bm25_rrf_weighted_judged,llm_judge_weighted_rrf,"baseline, judge_legal_terminology_rewrite, jud..."
6,process,Know Your Customer (KYC) regulations for indiv...,For the purposes of this Chapter a customer an...,0.013810,bm25_rrf_weighted_judged,llm_judge_weighted_rrf,"baseline, judge_legal_terminology_rewrite, jud..."
7,process,Know Your Customer (KYC) regulations for indiv...,An exemption does not apply if the reporting e...,0.013792,bm25_rrf_weighted_judged,llm_judge_weighted_rrf,"baseline, judge_legal_terminology_rewrite, jud..."
8,process,Know Your Customer (KYC) regulations for indiv...,"The reporting entity must, as soon as practic...",0.013722,bm25_rrf_weighted_judged,llm_judge_weighted_rrf,"baseline, judge_legal_terminology_rewrite, jud..."
9,process,Know Your Customer (KYC) regulations for indiv...,An AML/CTF program must include a procedure fo...,0.013575,bm25_rrf_weighted_judged,llm_judge_weighted_rrf,"baseline, judge_legal_terminology_rewrite, jud..."


,level,query,rel_text,score,method,query_variant,source_variants
0,process,Know Your Customer (KYC) regulations for indiv...,An exemption does not apply if the reporting e...,-0.404671,bm25_rrf_weighted_ce_judged,llm_judge_weighted_rrf,243
1,process,Know Your Customer (KYC) regulations for indiv...,An AML/CTF program must include a procedure fo...,-0.995963,bm25_rrf_weighted_ce_judged,llm_judge_weighted_rrf,22
2,process,Know Your Customer (KYC) regulations for indiv...,"The reporting entity must, as soon as practic...",-1.018026,bm25_rrf_weighted_ce_judged,llm_judge_weighted_rrf,8
3,process,Know Your Customer (KYC) regulations for indiv...,An AML/CTF program must include appropriate ri...,-1.227114,bm25_rrf_weighted_ce_judged,llm_judge_weighted_rrf,24
4,process,Know Your Customer (KYC) regulations for indiv...,An AML/CTF program must include appropriate ri...,-1.320819,bm25_rrf_weighted_ce_judged,llm_judge_weighted_rrf,25
5,process,Know Your Customer (KYC) regulations for indiv...,An AML/CTF program must include appropriate ri...,-1.396689,bm25_rrf_weighted_ce_judged,llm_judge_weighted_rrf,36
6,process,Know Your Customer (KYC) regulations for indiv...,An AML/CTF program must include a procedure fo...,-1.490105,bm25_rrf_weighted_ce_judged,llm_judge_weighted_rrf,19
7,process,Know Your Customer (KYC) regulations for indiv...,An AML/CTF program must include appropriate r...,-1.526971,bm25_rrf_weighted_ce_judged,llm_judge_weighted_rrf,21
8,process,Know Your Customer (KYC) regulations for indiv...,An AML/CTF program must include a procedure fo...,-1.717122,bm25_rrf_weighted_ce_judged,llm_judge_weighted_rrf,20
9,process,Know Your Customer (KYC) regulations for indiv...,If a reporting entity is unable to establish t...,-1.729013,bm25_rrf_weighted_ce_judged,llm_judge_weighted_rrf,28


## Inspect Judge Query Rewrites

The judge-refined variants are written to `output_with_agents_<use_case>.csv` in columns prefixed with `judge_`.

In [14]:
judge_columns = [
    "level",
    "query",
    "judge_legal_terminology_rewrite",
    "judge_regulatory_compliance_query",
    "judge_contract_clause_query",
    "judge_risk_scenario_query",
]

if agents_path.exists():
    agents_df = pd.read_csv(agents_path)
    available_judge_columns = [column for column in judge_columns if column in agents_df.columns]
    if len(available_judge_columns) == len(judge_columns):
        display(agents_df[judge_columns].head(10))
    else:
        print("Judge columns not found yet. Run the judged pipeline first.")
        display(agents_df.head(10))
else:
    print(f"Missing {agents_path}. Run the pipeline first.")

,level,query,judge_legal_terminology_rewrite,judge_regulatory_compliance_query,judge_contract_clause_query,judge_risk_scenario_query
0,process,Know Your Customer (KYC) regulations for indiv...,Establishing comprehensive legal frameworks an...,Regulatory requirements and obligations for fi...,Provisions and contractual terms related to cu...,Consequences and legal ramifications of non-co...
1,subprocess,"To create a new account, the required personal...",Establishment of customer accounts by financia...,Regulatory requirements and obligations for cu...,Account opening agreements detailing customer ...,Consequences of non-compliance with KYC/AML re...
2,subprocess,A customers identity is validated by verifying...,Mandatory procedures for the verification and ...,Regulatory requirements and statutory obligati...,Terms and conditions for customer identity ver...,Consequences of inadequate or failed customer ...
3,subprocess,A customer's identity is checked against a Pol...,Procedures for verifying customer identity aga...,"Obligations for customer due diligence, includ...",Contractual stipulations and obligations conce...,Consequences of inadequate customer identity v...
4,subprocess,"To verify an account, the accuracy of the cust...","Ascertainment of account authenticity, validat...","Obligations for customer account verification,...","Provisions concerning account verification, cu...",Consequences of inadequate account verificatio...
5,subprocess,"To manually review a new customer, a thorough ...",Due diligence investigation and verification o...,Legal requirements for customer due diligence ...,"""customer due diligence obligations, onboardin...",legal implications of inadequate customer due ...
6,subprocess,"To onboard a new customer, their account is cr...","Formalization of client engagement protocols, ...",Requirements for financial institutions regard...,Provisions for customer account opening proced...,Penalties for failing to adequately verify cus...
7,subprocess,"To welcome a new customer, an overview of the ...","Client onboarding protocols, disclosure requir...",Bank's legal obligations for customer onboardi...,"Customer onboarding agreement terms, service p...",Legal and operational risks associated with cu...
8,task,Collect all the necessary personal information...,Acquisition of essential client data and suppo...,Obligations and requirements for the collectio...,Contractual provisions and service level agree...,Consequences of inadequate customer identifica...
9,task,Enter the customer's information into the bank...,"Procedures for the collection, verification, a...",Obligations for financial institutions regardi...,Specific clauses detailing the customer's duty...,Consequences of inaccurate or incomplete custo...


## Evaluation Helpers

The first evaluation reports MAP plus average true positives and false negatives per query. This makes recall changes easy to inspect.

The second evaluation creates the paper-style Accuracy, Precision, Recall table for the judged method.

In [15]:
SOTA_BASE = project_root / "regulatory_relevance4process-D73C/SOTA_NLP_LIR"
LEVEL_LABELS = {
    "process": "level 1: process relevance",
    "subprocess": "level 2: sub-process relevance",
    "task": "level 3: task/event relevance",
}
LEVEL_GS_FILES = {
    "process": "process_level",
    "subprocess": "subprocess_level",
    "task": "event_level",
}

def clean_text(text):
    cleaned = str(text)
    cleaned = cleaned.replace("or\n\n\n", " ")
    cleaned = cleaned.replace("or\n\n", " ")
    cleaned = cleaned.replace("and\n\n\n", " ")
    cleaned = cleaned.replace("and\n\n", " ")
    cleaned = cleaned.replace("\n\n\n", " ")
    cleaned = cleaned.replace("\n\n", " ")
    cleaned = cleaned.replace("\n \n", " ")
    cleaned = cleaned.replace("\n", " ")
    return cleaned.strip()

def gold_path_for(use_case, level):
    return SOTA_BASE / f"output_ranking_input_eval/{use_case}/gold_standard/gs_{use_case}_{LEVEL_GS_FILES[level]}.xlsx"

def corpus_path_for(use_case):
    return SOTA_BASE / f"input_ranking/{use_case}/Input_corpus_{use_case}.xlsx"

def average_precision(predicted, relevant):
    if not relevant:
        return np.nan
    hits = 0
    precision_sum = 0.0
    seen = set()
    for rank, rel_text in enumerate(predicted, start=1):
        if rel_text in seen:
            continue
        seen.add(rel_text)
        if rel_text in relevant:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / len(relevant)

def load_sheet(sheet_name):
    try:
        return pd.read_excel(ranking_path, sheet_name=sheet_name)
    except ValueError:
        return pd.DataFrame(columns=["level", "query", "rel_text", "score"])

def evaluate_map_for_sheet(sheet_name):
    predictions = load_sheet(sheet_name).copy()
    if predictions.empty:
        return pd.DataFrame(columns=["sheet", "level", "MAP", "avg_true_positives", "avg_false_negatives"])

    predictions["query"] = predictions["query"].apply(clean_text)
    predictions["rel_text"] = predictions["rel_text"].apply(clean_text)

    rows = []
    for level in ["process", "subprocess", "task"]:
        level_predictions = predictions[predictions["level"] == level]
        gold = pd.read_excel(gold_path_for(USE_CASE, level)).copy()
        gold["query"] = gold["query"].apply(clean_text)
        gold["rel_text"] = gold["rel_text"].apply(clean_text)

        average_precisions = []
        true_positives = []
        false_negatives = []
        for query in sorted(gold["query"].unique()):
            relevant = set(gold[gold["query"] == query]["rel_text"].tolist())
            predicted = level_predictions[level_predictions["query"] == query]["rel_text"].tolist()
            predicted_set = set(predicted)
            average_precisions.append(average_precision(predicted, relevant))
            true_positives.append(len(relevant & predicted_set))
            false_negatives.append(len(relevant - predicted_set))

        rows.append(
            {
                "sheet": sheet_name,
                "level": level,
                "MAP": np.nanmean(average_precisions),
                "avg_true_positives": np.mean(true_positives),
                "avg_false_negatives": np.mean(false_negatives),
            }
        )
    return pd.DataFrame(rows)

def relevance_metrics_for_level(use_case, level, sheet_name):
    predictions = load_sheet(sheet_name).copy()
    if predictions.empty:
        return {"Acc.": np.nan, "Prec.": np.nan, "Rec.": np.nan}

    corpus = pd.read_excel(corpus_path_for(use_case))
    corpus_texts = set(corpus["requirement_text"].astype(str).apply(clean_text).tolist())

    gold = pd.read_excel(gold_path_for(use_case, level)).copy()
    gold["query"] = gold["query"].apply(clean_text)
    gold["rel_text"] = gold["rel_text"].apply(clean_text)

    predictions = predictions[predictions["level"] == level].copy()
    predictions["query"] = predictions["query"].apply(clean_text)
    predictions["rel_text"] = predictions["rel_text"].apply(clean_text)

    true_positives = 0
    false_positives = 0
    false_negatives = 0
    true_negatives = 0
    queries = sorted(set(gold["query"].tolist()) | set(predictions["query"].tolist()))

    for query in queries:
        gold_relevant = set(gold[gold["query"] == query]["rel_text"].tolist())
        predicted_relevant = set(predictions[predictions["query"] == query]["rel_text"].tolist())
        true_positives += len(gold_relevant & predicted_relevant)
        false_positives += len(predicted_relevant - gold_relevant)
        false_negatives += len(gold_relevant - predicted_relevant)
        true_negatives += len(corpus_texts - gold_relevant - predicted_relevant)

    accuracy_denominator = true_positives + false_positives + false_negatives + true_negatives
    precision_denominator = true_positives + false_positives
    recall_denominator = true_positives + false_negatives
    accuracy = (true_positives + true_negatives) / accuracy_denominator if accuracy_denominator else np.nan
    precision = true_positives / precision_denominator if precision_denominator else np.nan
    recall = true_positives / recall_denominator if recall_denominator else np.nan
    return {"Acc.": accuracy, "Prec.": precision, "Rec.": recall}

## MAP And Recall-Oriented Counts

This compares the judged BM25 stage and the judged finished pipeline stage.

In [16]:
map_results = pd.concat(
    [
        evaluate_map_for_sheet(JUDGED_BM25_SHEET),
        evaluate_map_for_sheet(JUDGED_CE_SHEET),
    ],
    ignore_index=True,
)

map_results

,sheet,level,MAP,avg_true_positives,avg_false_negatives
0,BM25_RRF_weighted_judged,process,0.209103,17.000000,14.000000
1,BM25_RRF_weighted_judged,subprocess,0.042594,2.000000,6.571429
2,BM25_RRF_weighted_judged,task,0.037520,0.789474,5.000000
3,BM25_RRF_weighted_CE_judged,process,0.197711,17.000000,14.000000
4,BM25_RRF_weighted_CE_judged,subprocess,0.070383,2.000000,6.571429
5,BM25_RRF_weighted_CE_judged,task,0.060061,0.789474,5.000000


## Paper-Style Table

This table reports Accuracy, Precision, and Recall for the judged final pipeline output. The format mirrors the paper table, but focuses on the current judged approach.

In [17]:
paper_rows = []
for level in ["process", "subprocess", "task"]:
    metrics = relevance_metrics_for_level(USE_CASE, level, JUDGED_CE_SHEET)
    row = {
        ("process level", ""): LEVEL_LABELS[level],
        ("method", ""): "BM25+CE + weighted query diversification + LLM judge",
        (USE_CASE.upper(), "Acc."): metrics["Acc."],
        (USE_CASE.upper(), "Prec."): metrics["Prec."],
        (USE_CASE.upper(), "Rec."): metrics["Rec."],
    }
    paper_rows.append(row)

paper_table = pd.DataFrame(paper_rows)
paper_table.columns = pd.MultiIndex.from_tuples(paper_table.columns)
metric_columns = [column for column in paper_table.columns if column[1] in {"Acc.", "Prec.", "Rec."}]
paper_table[metric_columns] = paper_table[metric_columns].round(2)

paper_table

process level  \
                                    
0      level 1: process relevance   
1  level 2: sub-process relevance   
2   level 3: task/event relevance   

                                              method   UC2              
                                                      Acc. Prec.  Rec.  
0  BM25+CE + weighted query diversification + LLM...  0.74  0.17  0.55  
1  BM25+CE + weighted query diversification + LLM...  0.89  0.07  0.23  
2  BM25+CE + weighted query diversification + LLM...  0.94  0.05  0.14

## Export Results

This exports the MAP summary and the paper-style metrics for later reporting.

In [18]:
export_path = project_root / f"llm_judge_evaluation_{USE_CASE}.xlsx"

with pd.ExcelWriter(export_path) as writer:
    map_results.to_excel(writer, sheet_name="map_summary", index=False)
    paper_table.to_excel(writer, sheet_name="paper_table")
    judged_bm25.to_excel(writer, sheet_name="judged_bm25", index=False)
    judged_ce.to_excel(writer, sheet_name="judged_ce", index=False)

print(f"Exported {export_path}")

Exported /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval/llm_judge_evaluation_uc2.xlsx
